# 🏗️ RAG local completo con interfaz Gradio

**Laboratorio de PLN: Analítica, Textos y Cultura**
Matías Barreto — IFTS24, 2026

**Proyecto integrador — 90 minutos**

---

## Objetivo

Ensamblar un sistema RAG completo que funcione desde la terminal: cargás tus propios PDFs, los vectorizás, los almacenás en ChromaDB, consultás con un LLM local y todo queda empaquetado en una interfaz Gradio lista para deployar en HuggingFace Spaces.

## Al terminar este notebook vas a poder:

1. Construir un pipeline RAG end-to-end con documentos propios, sin depender de ninguna API externa.
2. Crear una interfaz de usuario con Gradio que permita subir PDFs y hacer preguntas al sistema.
3. Generar el código `app.py` y `requirements.txt` listos para desplegar en HuggingFace Spaces.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Pipeline end-to-end** | Sistema completo donde cada pieza pasa su salida a la siguiente sin intervención manual. |
| **ChromaDB persistente** | Base de datos vectorial que guarda los embeddings en disco para no recalcularlos en cada ejecución. |
| **Gradio** | Librería que convierte funciones Python en interfaces web interactivas con pocas líneas de código. |
| **HuggingFace Spaces** | Plataforma gratuita que permite publicar aplicaciones Gradio en la web con un clic. |
| **Serverless inference** | Modelo de IA que corre en la infraestructura de HuggingFace sin que vos alojes nada. |
| **app.py** | Archivo principal de una aplicación Gradio — el punto de entrada que Spaces busca para arrancar. |

## Analogía

Pensá en este proyecto como armar una biblioteca con un asistente propio. Primero instalás las estanterías (ChromaDB), después digitalizás y catalogás tus libros (cargás y vectorizás los PDFs), luego contratás al asistente que sabe leer el catálogo (el LLM con RAG), y por último le ponés un mostrador al frente para que cualquier persona pueda hacer preguntas sin saber nada de lo que hay atrás (Gradio).

## Dónde vive esto en el mundo real

Este patrón — documentos propios + LLM local + interfaz web — es exactamente lo que usan equipos de RRHH para consultar políticas internas, estudios jurídicos para buscar jurisprudencia, y equipos de investigación para navegar miles de papers. La diferencia con los sistemas comerciales es que acá todo corre en tu máquina y los documentos no salen a ningún servidor externo.

In [ ]:
# Instalamos todas las dependencias del proyecto integrador
!uv pip install langchain langchain-community langchain-chroma langchain-ollama \
    langchain-text-splitters langchain-core \
    chromadb sentence-transformers \
    gradio pypdf python-dotenv -q

print("✓ Dependencias instaladas")
print("  Recordá tener Ollama corriendo: ollama serve")

## ⚙️ Configuración del entorno

Antes de cargar documentos o configurar el LLM, detectamos el hardware disponible.
Eso nos permite elegir el modelo y los parámetros más adecuados sin sobrecalentar la máquina.

In [ ]:
import platform
import subprocess

# Detectamos el sistema operativo y la arquitectura del procesador
sistema = platform.system()     # 'Darwin', 'Linux', 'Windows'
maquina = platform.machine()    # 'arm64' (Apple Silicon), 'x86_64'

# Verificamos si hay GPU NVIDIA disponible (solo Linux/Windows)
tiene_cuda = False
if sistema in ("Linux", "Windows"):
    try:
        resultado = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        tiene_cuda = resultado.returncode == 0
    except FileNotFoundError:
        tiene_cuda = False

# Determinamos el perfil de hardware y configuramos el modelo en consecuencia
es_apple_silicon = (sistema == "Darwin" and maquina == "arm64")

if es_apple_silicon:
    PERFIL_HARDWARE = "Apple Silicon (Metal)"
    MODEL_NAME      = "gemma4:e2b"   # liviano, corre bien en M-series
    NUM_CTX         = 4096
elif tiene_cuda:
    PERFIL_HARDWARE = "NVIDIA GPU (CUDA)"
    MODEL_NAME      = "granite4:latest"
    NUM_CTX         = 8192
else:
    PERFIL_HARDWARE = "CPU (sin GPU dedicada)"
    MODEL_NAME      = "gemma4:e2b"   # el más conservador
    NUM_CTX         = 1024

print(f"Sistema:          {sistema} — {maquina}")
print(f"Perfil detectado: {PERFIL_HARDWARE}")
print(f"Modelo elegido:   {MODEL_NAME}")
print(f"Contexto (tokens):{NUM_CTX}")

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Carpeta donde el sistema buscará los PDFs por defecto
CARPETA_DATOS = os.path.join(os.getcwd(), "CARPETA_DATOS")

# Directorio donde ChromaDB guardará los vectores en disco
DIRECTORIO_CHROMA = os.path.join(os.getcwd(), "chroma_proyecto")

print(f"Carpeta de PDFs:  {CARPETA_DATOS}")
print(f"Base vectorial:   {DIRECTORIO_CHROMA}")

## Paso 1 — Carga de documentos PDF

El `PyPDFLoader` extrae texto página por página y conserva metadatos como el nombre del archivo y el número de página. Eso nos permite mostrar las fuentes exactas cuando el sistema responde.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Listamos todos los PDFs en la carpeta de datos
archivos_pdf = sorted([
    f for f in os.listdir(CARPETA_DATOS)
    if f.lower().endswith(".pdf")
])

if not archivos_pdf:
    print("⚠️  No hay PDFs en la carpeta.")
    print(f"   Copiá al menos un PDF en: {CARPETA_DATOS}")
else:
    print(f"Archivos encontrados: {len(archivos_pdf)}")

In [ ]:
# Cargamos página por página y acumulamos en una lista
documentos_crudos = []

for nombre_archivo in archivos_pdf:
    ruta_completa = os.path.join(CARPETA_DATOS, nombre_archivo)
    loader = PyPDFLoader(ruta_completa)
    paginas = loader.load()
    documentos_crudos.extend(paginas)
    print(f"  ✓ {nombre_archivo} — {len(paginas)} páginas")

print(f"\n✓ Total: {len(documentos_crudos)} páginas cargadas")

## Paso 2 — Fragmentación de documentos

Un PDF completo es demasiado grande para inyectar en el prompt. Lo dividimos en fragmentos que:
- Sean lo suficientemente pequeños para caber en el contexto del LLM
- Sean lo suficientemente grandes para tener sentido por sí solos
- Se superpongan un poco (overlap) para no cortar ideas en el límite

### ✎ Para pensar
- ¿Qué pasaría si `chunk_size` fuera de 50 caracteres? ¿Y de 5000?
- ¿Por qué necesitamos `chunk_overlap` y no podemos simplemente cortar sin superposición?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter intenta dividir por párrafos primero,
# luego por oraciones, luego por espacios — respetando la coherencia semántica
divisor = RecursiveCharacterTextSplitter(
    chunk_size=600,        # máximo 600 caracteres por fragmento
    chunk_overlap=80,      # 80 caracteres de solapamiento entre fragmentos
    separators=["\n\n", "\n", ". ", " "]
)

fragmentos = divisor.split_documents(documentos_crudos)

print(f"✓ Fragmentación completada")
print(f"  Documentos originales: {len(documentos_crudos)} páginas")
print(f"  Fragmentos generados:  {len(fragmentos)}")

# Mostramos un fragmento de ejemplo para verificar
print(f"\n◈ Ejemplo de fragmento:")
print(f"  Fuente: {fragmentos[0].metadata.get('source', 'desconocida')}")
print(f"  Página: {fragmentos[0].metadata.get('page', '?')}")
print(f"  Texto:  {fragmentos[0].page_content[:200]}...")

## Paso 3 — Embeddings y ChromaDB persistente

Convertiamos el texto en vectores numéricos (embeddings) que representan el significado semántico. Luego los guardamos en ChromaDB en disco — así la próxima vez que ejecutemos el notebook no necesitamos recalcularlos.

**¿Por qué persistir?** Vectorizar 100 páginas puede tardar varios minutos. Si guardamos los vectores, la próxima sesión carga en segundos.

In [ ]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# multilingual-e5-large corre localmente sin consumir API
# Entiende bien el español rioplatense y múltiples idiomas
modelo_embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

print("✓ Modelo de embeddings configurado")
print("  intfloat/multilingual-e5-large — local, sin API")

In [ ]:
from langchain_chroma import Chroma

# Creamos (o recargamos si ya existe) la base vectorial persistente
# Si el directorio chroma_proyecto ya existe, Chroma lo carga sin recalcular
vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings,
    collection_name="proyecto_rag",
    persist_directory=DIRECTORIO_CHROMA
)

print(f"✓ Base vectorial lista")
print(f"  Directorio: {DIRECTORIO_CHROMA}")
print(f"  Fragmentos indexados: {len(fragmentos)}")

## Paso 4 — LLM local con Ollama

Usamos el modelo que detectamos al inicio. El parámetro `num_ctx` le dice a Ollama cuántos tokens puede usar como contexto — más contexto permite fragmentos más largos pero requiere más memoria.

In [ ]:
from langchain_ollama import OllamaLLM

# Conectamos con el servidor Ollama que corre localmente en el puerto 11434
# temperature=0.1 para respuestas precisas y reproducibles
llm = OllamaLLM(
    model=MODEL_NAME,
    temperature=0.1,
    num_ctx=NUM_CTX,
)

# Verificamos que Ollama responde antes de armar el pipeline completo
respuesta_prueba = llm.invoke("Respondé solo con 'ok' si estás funcionando.")
print(f"✓ Ollama responde: {respuesta_prueba.strip()}")
print(f"  Modelo activo: {MODEL_NAME}")

## Paso 5 — Pipeline RAG con LCEL

Ensamblamos el pipeline completo usando LCEL (LangChain Expression Language). La cadena conecta retriever → prompt → LLM → parser en una sola expresión que podemos invocar con `.invoke(pregunta)`.

### ✎ Para pensar
- ¿Qué hace el `format_docs`? ¿Por qué necesitamos esa función intermedia?
- ¿Qué cambiaría si `k=5` en lugar de `k=3`?

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# El retriever busca los 3 fragmentos más similares a la pregunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def formatear_documentos(docs):
    # Une los fragmentos recuperados en un solo bloque de texto
    return "\n\n".join(doc.page_content for doc in docs)

TEMPLATE = """Sos un asistente que responde preguntas basándose ÚNICAMENTE en los documentos proporcionados.
Si la respuesta no está en los documentos, decilo claramente: no la inventes.

Documentos:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ Pipeline RAG configurado")
print("  Flujo: pregunta → ChromaDB (k=3) → prompt → Ollama → respuesta")

In [ ]:
# Probamos el pipeline con una pregunta de prueba antes de abrir la interfaz
pregunta_test = "¿De qué tratan los documentos que cargaste?"

respuesta = pipeline_rag.invoke(pregunta_test)

print(f"Pregunta: {pregunta_test}")
print(f"\nRespuesta:")
print(respuesta)

## Paso 6 — Interfaz Gradio

### Analogía

Gradio es como ponerle una caja de entrada y una pantalla de salida a tu pipeline. En lugar de escribir código Python para cada consulta, el usuario final interactúa con botones, campos de texto y un chat — sin necesidad de saber programar.

### Dónde vive esto en el mundo real

Gradio es el estándar de facto para prototipos de IA. La mayoría de los demos en HuggingFace Spaces están construidos con él. Aprender a armar una interfaz Gradio es directamente transferible al trabajo en equipos de datos e IA.

In [ ]:
import gradio as gr

# ─── Funciones que conectan la interfaz con el pipeline ───────────────────────

def cargar_pdfs_interfaz(archivos):
    """Recibe archivos subidos desde Gradio, los indexa y devuelve un mensaje de estado."""
    if not archivos:
        return "No se seleccionaron archivos."

    nuevas_paginas = []
    nombres = []

    for archivo in archivos:
        loader = PyPDFLoader(archivo.name)
        paginas = loader.load()
        nuevas_paginas.extend(paginas)
        nombres.append(Path(archivo.name).name)

    nuevos_fragmentos = divisor.split_documents(nuevas_paginas)

    # Agregamos al vectorstore existente sin reiniciarlo
    vectorstore.add_documents(nuevos_fragmentos)

    return f"✓ Archivos cargados: {', '.join(nombres)}\n✓ Fragmentos indexados: {len(nuevos_fragmentos)}"


def responder_pregunta(pregunta, historial):
    """Invoca el pipeline RAG y devuelve la respuesta junto con las fuentes consultadas."""
    if not pregunta.strip():
        return historial, ""

    respuesta = pipeline_rag.invoke(pregunta)

    # Recuperamos los fragmentos fuente para mostrarlos
    fragmentos_fuente = retriever.invoke(pregunta)

    lineas_fuente = []
    for frag in fragmentos_fuente:
        fuente = Path(frag.metadata.get("source", "desconocida")).name
        pagina = frag.metadata.get("page", "?")
        lineas_fuente.append(f"• {fuente} (pág. {pagina})")

    # Gradio 5 usa formato de mensajes (OpenAI-style), no tuplas
    historial = historial + [
        {"role": "user",      "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    texto_fuentes = "\n".join(lineas_fuente)

    return historial, texto_fuentes


print("✓ Funciones de interfaz definidas")

In [ ]:
# ─── Construcción de la interfaz ──────────────────────────────────────────────

with gr.Blocks(title="RAG Local — IFTS24", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# RAG Local con Ollama y ChromaDB")
    gr.Markdown("**Laboratorio de PLN — IFTS24, 2026**")

    with gr.Tab("📄 Cargar documentos"):
        gr.Markdown("Subí uno o más PDFs para agregarlos a la base de conocimiento.")
        upload_component = gr.File(
            label="Seleccioná tus PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )
        boton_cargar = gr.Button("Indexar documentos", variant="primary")
        estado_carga = gr.Textbox(label="Estado", interactive=False, lines=3)
        boton_cargar.click(
            fn=cargar_pdfs_interfaz,
            inputs=[upload_component],
            outputs=[estado_carga]
        )

    with gr.Tab("💬 Hacer preguntas"):
        # Gradio 6.x: el formato role/content es el único disponible, sin parámetro type
        chatbot_componente = gr.Chatbot(label="Conversación", height=400)
        with gr.Row():
            pregunta_componente = gr.Textbox(
                label="Tu pregunta",
                placeholder="¿Qué dice el documento sobre...?",
                scale=4
            )
            boton_preguntar = gr.Button("Preguntar", variant="primary", scale=1)
        fuentes_componente = gr.Textbox(
            label="Fragmentos consultados",
            interactive=False,
            lines=3
        )
        boton_preguntar.click(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
        pregunta_componente.submit(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )

demo.launch(share=False)

### ✎ Para pensar

- Cuando subís un PDF nuevo desde la pestaña "Cargar documentos", ¿qué pasa con los fragmentos que ya estaban en ChromaDB? ¿Se borran o se acumulan?
- La interfaz muestra las fuentes consultadas. ¿Por qué es importante mostrar esa información en lugar de solo la respuesta?
- ¿Qué pasaría si cargás un PDF en inglés y hacés preguntas en español? ¿El sistema podría responder bien?

---

## 🌐 Hacia HuggingFace Spaces

Hasta acá el sistema corre en tu máquina con Ollama. Para publicarlo en HuggingFace Spaces necesitamos dos cambios:

1. **LLM**: Ollama no corre en Spaces — usamos la API de inferencia serverless de HuggingFace (gratis con una cuenta).
2. **ChromaDB**: usamos modo en memoria (sin `persist_directory`) porque el almacenamiento en Spaces es efímero.

Las celdas siguientes generan el `app.py` y el `requirements.txt` listos para subir.

### Pasos para publicar

1. Creá una cuenta en [huggingface.co](https://huggingface.co) si no tenés.
2. Generá un token en `Settings → Access Tokens` (con permisos de escritura).
3. En tu perfil, creá un nuevo Space → tipo **Gradio**.
4. Subí los archivos `app.py` y `requirements.txt` generados abajo.
5. En la sección `Settings → Repository Secrets` del Space, creá el secreto `HF_TOKEN` con tu token.
6. El Space se construye automáticamente y queda disponible en una URL pública.

In [ ]:
%%writefile app.py
# ─────────────────────────────────────────────────────────────────────────────
# app.py — RAG con HuggingFace Inference API y Gradio
# Para HuggingFace Spaces: configurá el secreto HF_TOKEN en Settings.
# Uso local:  HF_TOKEN=tu_token python app.py
# ─────────────────────────────────────────────────────────────────────────────

import os
from pathlib import Path
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint

# ─── Configuración ────────────────────────────────────────────────────────────

HF_TOKEN  = os.environ.get("HF_TOKEN", "")
MODEL_ID  = "meta-llama/Llama-3.2-3B-Instruct"  # gratuito con token HF

if not HF_TOKEN:
    raise ValueError("Configurá el secreto HF_TOKEN en el Space.")

# ─── Embeddings locales (corren en la CPU del Space) ──────────────────────────

modelo_embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

# ─── ChromaDB en memoria (sin disco para Spaces) ──────────────────────────────

vectorstore = Chroma(
    collection_name="proyecto_rag_spaces",
    embedding_function=modelo_embeddings
)

# ─── Divisor de texto ─────────────────────────────────────────────────────────

divisor = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)

# ─── LLM via HuggingFace Serverless Inference ─────────────────────────────────

llm = HuggingFaceEndpoint(
    repo_id=MODEL_ID,
    temperature=0.1,
    max_new_tokens=512,
    huggingfacehub_api_token=HF_TOKEN
)

# ─── Pipeline RAG ─────────────────────────────────────────────────────────────

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def formatear_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)

TEMPLATE = """Respondé la siguiente pregunta usando ÚNICAMENTE los documentos proporcionados.
Si la respuesta no está, decilo claramente.

Documentos:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ─── Funciones de la interfaz ─────────────────────────────────────────────────

def cargar_pdfs_interfaz(archivos):
    if not archivos:
        return "No se seleccionaron archivos."
    nuevas_paginas = []
    nombres = []
    for archivo in archivos:
        loader = PyPDFLoader(archivo.name)
        paginas = loader.load()
        nuevas_paginas.extend(paginas)
        nombres.append(Path(archivo.name).name)
    nuevos_fragmentos = divisor.split_documents(nuevas_paginas)
    vectorstore.add_documents(nuevos_fragmentos)
    return f"✓ Archivos: {', '.join(nombres)}\n✓ Fragmentos: {len(nuevos_fragmentos)}"

def responder_pregunta(pregunta, historial):
    if not pregunta.strip():
        return historial, ""
    respuesta = pipeline_rag.invoke(pregunta)
    fragmentos_fuente = retriever.invoke(pregunta)
    lineas_fuente = []
    for frag in fragmentos_fuente:
        fuente = Path(frag.metadata.get("source", "desconocida")).name
        pagina = frag.metadata.get("page", "?")
        lineas_fuente.append(f"• {fuente} (pág. {pagina})")
    historial = historial + [
        {"role": "user",      "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return historial, "\n".join(lineas_fuente)

# ─── Interfaz Gradio ──────────────────────────────────────────────────────────

with gr.Blocks(title="RAG Local — IFTS24", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# RAG con HuggingFace Spaces")
    gr.Markdown("**Laboratorio de PLN — IFTS24, 2026**")

    with gr.Tab("📄 Cargar documentos"):
        upload_component = gr.File(
            label="Seleccioná tus PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )
        boton_cargar = gr.Button("Indexar documentos", variant="primary")
        estado_carga = gr.Textbox(label="Estado", interactive=False, lines=3)
        boton_cargar.click(
            fn=cargar_pdfs_interfaz,
            inputs=[upload_component],
            outputs=[estado_carga]
        )

    with gr.Tab("💬 Hacer preguntas"):
        chatbot_componente = gr.Chatbot(label="Conversación", height=400)
        with gr.Row():
            pregunta_componente = gr.Textbox(
                label="Tu pregunta",
                placeholder="¿Qué dice el documento sobre...?",
                scale=4
            )
            boton_preguntar = gr.Button("Preguntar", variant="primary", scale=1)
        fuentes_componente = gr.Textbox(
            label="Fragmentos consultados",
            interactive=False,
            lines=3
        )
        boton_preguntar.click(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
        pregunta_componente.submit(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )

demo.launch()

In [ ]:
%%writefile requirements.txt
# Proyecto Final — RAG con Gradio
# Laboratorio de PLN — IFTS24, 2026
#
# HuggingFace Spaces instala esto automáticamente.
# Localmente: uv pip install -r requirements.txt

# Pipeline RAG
langchain>=1.0.0
langchain-community>=0.4.0
langchain-chroma>=1.0.0
langchain-text-splitters>=1.0.0
langchain-core>=1.0.0
langchain-huggingface>=0.1.0

# Base vectorial y embeddings
chromadb>=0.6.0
sentence-transformers>=3.0.0

# Carga de PDFs
pypdf>=5.0.0

# Interfaz
gradio>=5.0.0

In [ ]:
# Verificamos que se generaron correctamente
import os

for archivo in ["app.py", "requirements.txt"]:
    if os.path.exists(archivo):
        tamanio = os.path.getsize(archivo)
        print(f"  ✓ {archivo} — {tamanio} bytes")
    else:
        print(f"  ✗ {archivo} no encontrado")

print("\nEstos dos archivos son todo lo que necesitás subir a tu Space.")

## ⛰️ Cierre del proyecto integrador

### Lo que construiste

| Pieza | Qué hace | Tecnología |
|---|---|---|
| Cargador de PDFs | Extrae texto página por página | PyPDFLoader |
| Divisor de texto | Fragmenta en chunks con overlap | RecursiveCharacterTextSplitter |
| Embeddings | Vectoriza texto localmente | multilingual-e5-large |
| Base vectorial | Almacena y busca fragmentos por similitud | ChromaDB (persistente) |
| LLM local | Genera respuestas fundamentadas | Ollama |
| Pipeline RAG | Conecta todo con LCEL | LangChain |
| Interfaz | Hace el sistema accesible sin código | Gradio |
| Deploy | Publica en la web | HuggingFace Spaces |

### La diferencia entre local y Spaces

| Aspecto | Local | HuggingFace Spaces |
|---|---|---|
| LLM | Ollama (tu hardware) | HuggingFace Inference API |
| ChromaDB | Persistente en disco | En memoria (efímera) |
| Acceso | Solo tu máquina | URL pública |
| Costo | Sin API, solo electricidad | Gratis con cuenta HF |

### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas:

1. ¿Qué pieza del pipeline te pareció más difícil de entender? ¿Ya la pudiste encajar?
2. Si tuvieras que presentar este sistema ante alguien de RRHH de tu empresa, ¿cómo lo explicarías en 30 segundos sin mencionar "embeddings" ni "vectores"?
3. ¿Qué problema real de tu contexto laboral o de estudio resolverías con este sistema?